# 05 - Evaluation and Baseline Comparison

This notebook compares the OR-Tools assignment against random, greedy, and current-assignment baselines using the same sampled employees and department capacities.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
candidate_roots = [current, current.parent, current.parent.parent]
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'smartassign_pipeline.py').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root.')

sys.path.insert(0, str(ROOT / 'src'))
from smartassign_pipeline import (
    FIGURES_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    ModelBundle,
    build_score_matrix,
    compare_assignment_methods,
    compute_department_capacities,
    compute_department_profiles,
    ensure_output_directories,
    get_counterfactual_profile_columns,
    load_raw_data,
    save_csv,
    split_train_test,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)

df = load_raw_data()
bundle = ModelBundle.load(MODELS_DIR / 'best_model_bundle.joblib')
train_x, test_x, train_y, test_y = split_train_test(df, bundle.feature_columns)
sample_df = df.loc[test_x.index].copy().sample(n=min(50, len(test_x)), random_state=42)

profile_columns = get_counterfactual_profile_columns(bundle.feature_set_name, df)
department_profiles = compute_department_profiles(df, profile_columns)
score_matrix = build_score_matrix(bundle, sample_df, department_profiles, profile_columns)
capacities = compute_department_capacities(df, len(sample_df))
comparison, assignments = compare_assignment_methods(sample_df, score_matrix, capacities)

display(comparison)
save_csv(comparison, RESULTS_DIR / 'assignment_comparison.csv')

plt.figure(figsize=(10, 5))
plt.bar(comparison['method'], comparison['total_predicted_score'], color='#5B9BD5')
plt.ylabel('Total Predicted Score')
plt.title('Assignment Comparison by Total Predicted Score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_assignment_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

optimal_total = float(comparison.loc[comparison['method'] == 'optimal_ortools', 'total_predicted_score'].iloc[0])
current_total = float(comparison.loc[comparison['method'] == 'current_assignment', 'total_predicted_score'].iloc[0])
improvement = optimal_total - current_total

summary_lines = [
    f'The OR-Tools assignment achieves the highest total predicted score at {optimal_total:.2f}.',
    f'Compared with the current assignment baseline, the model-driven improvement is {improvement:.2f} predicted-score points.',
    'Random and greedy baselines are included to show that the optimization step adds value beyond simple heuristics.',
    'The proxy true total score is constant across methods because the sampled employees are the same; it is reported only as a reference point.',
]
display(Markdown('### Evaluation Summary\n' + '\n'.join(f'- {line}' for line in summary_lines)))